# Lance DB - a vector database for LLM applications

In [2]:
import lancedb

db = lancedb.connect(uri="vector_database")

db.uri

'c:\\Users\\jonat\\OneDrive\\Skrivbord\\Everything\\Utveckling\\ai_engineer\\AI_engineering_jonathan_hansson\\code_alongs\\13_LanceDB\\vector_database'

## Create a table

In [3]:
import json

with open("data/animals_text_embeddings.json", "r") as file:
    data = json.loads(file.read())

data

[{'text': 'A small brown dog running.', 'vector': [0.12, 0.85, 0.33]},
 {'text': 'A cat resting quietly on a sofa.', 'vector': [0.4, 0.91, 0.1]},
 {'text': 'A large gray elephant drinking water.',
  'vector': [0.88, 0.22, 0.55]},
 {'text': 'A fast cheetah sprinting across the savannah.',
  'vector': [0.95, 0.12, 0.72]},
 {'text': 'A colorful parrot perched on a branch.',
  'vector': [0.25, 0.66, 0.81]},
 {'text': 'A frog sitting on a lily pad.', 'vector': [0.14, 0.44, 0.27]}]

In [4]:
table_animals = db.create_table("animals_text", exist_ok=True, data=data)

table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [5]:
more_data = [
    {"text": "A panda eating bamboo peacefully.", "vector": [0.51, 0.37, 0.82]},
    {"text": "A lion roaring loudly on a rock.", "vector": [0.93, 0.18, 0.41]}
]

table_animals.add(more_data)

AddResult(version=4)

In [6]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


## Create an empty table and then delete it 

In [7]:
from lancedb.pydantic import LanceModel

# A pydantic Model base class that can be converted to a LanceDB table

class JokeSchema(LanceModel):
    joke: str
    rating: int

db.create_table(name="jokes", schema=JokeSchema, exist_ok=True)

LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='c:\\Users\\jonat\\OneDrive\\Skrivbord\\Everything\\Utveckling\\ai_engineer\\AI_engineering_jonathan_hansson\\code_alongs\\13_LanceDB\\vector_database'))

In [8]:
db.drop_table("jokes")

In [9]:
db.list_tables()

ListTablesResponse(tables=['animals_text'], page_token=None)

## Open existing table

In [10]:
db.open_table("animals_text")

LanceTable(name='animals_text', version=4, _conn=LanceDBConnection(uri='c:\\Users\\jonat\\OneDrive\\Skrivbord\\Everything\\Utveckling\\ai_engineer\\AI_engineering_jonathan_hansson\\code_alongs\\13_LanceDB\\vector_database'))

## VECTOR SEARCH IN LANCEDB

In [11]:
table_animals.to_pandas()

,text,vector
0,A small brown dog running.,"[0.12, 0.85, 0.33]"
1,A cat resting quietly on a sofa.,"[0.4, 0.91, 0.1]"
2,A large gray elephant drinking water.,"[0.88, 0.22, 0.55]"
3,A fast cheetah sprinting across the savannah.,"[0.95, 0.12, 0.72]"
4,A colorful parrot perched on a branch.,"[0.25, 0.66, 0.81]"
5,A frog sitting on a lily pad.,"[0.14, 0.44, 0.27]"
6,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
7,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"
8,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]"
9,A lion roaring loudly on a rock.,"[0.93, 0.18, 0.41]"


In [12]:
query_vector = [0.5, 0.2, 0.9]

table_animals.search(query_vector).limit(3).to_pandas()

,text,vector,_distance
0,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
1,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354
2,A panda eating bamboo peacefully.,"[0.51, 0.37, 0.82]",0.0354


## Embedding API

- idea: want to put in text -> and it will automatically generate vector embeddings
- idea: want to put in images -> and it will automatically generate vector embeddings
- calculate closest distances

In [37]:
from lancedb.pydantic import LanceModel, Vector
from lancedb.embeddings import get_registry

model = get_registry().get("gemini-text").create(name="gemini-embedding-001")
type(model)

lancedb.embeddings.gemini_text.GeminiText

In [14]:
import numpy as np
from dotenv import load_dotenv

load_dotenv()

hello_embedding = np.array(model.generate_embeddings("hello"))

hello_embedding.shape

c:\Users\jonat\OneDrive\Skrivbord\Everything\Utveckling\ai_engineer\AI_engineering_jonathan_hansson\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\jonat\AppData\Local\Programs\Python\Python313\Lib\importlib\__init__.py:88: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  return _bootstrap._gcd_import(name[level:], package, level)


(5, 3072)

In [17]:
class JokeModel(LanceModel):
    joke: str = model.SourceField()
    vector: Vector(3072) = model.VectorField()

table_jokes = db.create_table("jokes", schema=JokeModel, exist_ok=True)
table_jokes


LanceTable(name='jokes', version=1, _conn=LanceDBConnection(uri='c:\\Users\\jonat\\OneDrive\\Skrivbord\\Everything\\Utveckling\\ai_engineer\\AI_engineering_jonathan_hansson\\code_alongs\\13_LanceDB\\vector_database'))

In [28]:
import pandas as pd

with open("data/jokes.json", "r", encoding="utf-8") as file:
    jokes_data = json.loads(file.read())

df = pd.DataFrame(jokes_data).rename({"jokes": "joke"}, axis=1)

df.head(5)



,joke
0,Parallel lines have so much in common—it’s sad...
1,"ETL stands for “Extract, Transform, Leave for ..."
2,What do you call a snake that runs your script...
3,"Gold walks into a bar. The bartender says, “Au..."
4,C# devs don’t argue; they just throw exceptions.


add data to table

In [29]:
table_jokes.add(df)

AddResult(version=2)

In [31]:
table_jokes.head(2)

pyarrow.Table
joke: string not null
vector: fixed_size_list<item: float>[3072]
  child 0, item: float
----
joke: [["Parallel lines have so much in common—it’s sad they’ll never meet.","ETL stands for “Extract, Transform, Leave for the next person.”"]]
vector: [[[-0.024001757,0.01247358,-0.024144737,-0.06704516,0.017059995,...,0.011402721,-0.017770408,0.011606138,0.0004488597,0.012175956],[-0.015356114,0.0211365,-0.021389864,-0.07957475,0.008829045,...,0.023671096,0.00070166244,0.003873497,0.0006493248,0.01587487]]]

In [35]:
table_jokes.to_pandas()["vector"][0]

array([-0.02400176,  0.01247358, -0.02414474, ...,  0.01160614,
        0.00044886,  0.01217596], shape=(3072,), dtype=float32)

## Perform vector search

In [45]:
table_jokes.search("give me nature related jokes").limit(8).to_pandas()

,joke,vector,_distance
0,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.592044
1,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.693084
2,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.723715
3,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.725794
4,The C# compiler walked into a bar. The bartend...,"[-0.01868987, 0.018796643, -0.009748903, -0.07...",0.750098
5,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0...",0.761704
6,Why did the Python programmer get bitten? Beca...,"[-0.021944793, 0.0030636177, -0.019837778, -0....",0.770420
7,Why’s 6 afraid of 7? Because 7 8 9.,"[-0.040820926, 0.0074244644, -0.02972158, -0.0...",0.775439


## Hybrid search

Combines traditional keyword-based search with vector similarity search

In [39]:
# To enable keyword search, we need to create an index on the 'joke' column

table_jokes.create_fts_index("joke", replace=True)

In [44]:
from lancedb import rerankers

reranker = rerankers.RRFReranker()

results = table_jokes.search(
    "give me nature related jokes",
    query_type="hybrid",
    vector_column_name="vector",
    fts_columns="joke"
).rerank(reranker=reranker).limit(8).to_pandas()

results

,joke,vector,_relevance_score
0,Why do data engineers hate nature? Too many un...,"[-0.027916763, 0.0047387416, -0.018934403, -0....",0.032522
1,I told a chemistry joke… there was no reaction.,"[-0.022922393, 0.017959604, -0.029222224, -0.0...",0.032018
2,"Gold walks into a bar. The bartender says, “Au...","[-0.024867292, 0.013314825, -0.016261652, -0.0...",0.016129
3,Why did the chemist ground his kids? Because t...,"[-0.023257235, 0.016145445, -0.029016329, -0.0...",0.015873
4,The C# compiler walked into a bar. The bartend...,"[-0.01868987, 0.018796643, -0.009748903, -0.07...",0.015385
5,What do you call a snake that runs your script...,"[-0.01761013, 0.0031474787, -0.015632002, -0.0...",0.015152
6,Why did the Python programmer get bitten? Beca...,"[-0.021944793, 0.0030636177, -0.019837778, -0....",0.014925
7,Why’s 6 afraid of 7? Because 7 8 9.,"[-0.040820926, 0.0074244644, -0.02972158, -0.0...",0.014706


## Rule of thumb

- for exact matching -> FTS (full text search)
- meaning based matching -> vector search
- both -> hybrid search